# 01 — Data Exploration

**Job:** understand the raw ClinVar table and produce the first clean checkpoint.

**Contract:** reads `data/raw/variant_summary.txt.gz` → writes
`data/interim/labeled_snvs.parquet`.

This notebook *narrates* what the data looks like. The filtering and
label-cleaning rules themselves live in `load_clinvar.py` (tested, reusable);
here we call that module and then explore its output.

## 01 — Imports and paths

In [ ]:
"""01_data_exploration.ipynb — Preliminary exploration of ClinVar missense variants."""
%load_ext autoreload
%autoreload 2

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from load_clinvar import load_and_label_clinvar, print_funnel

PROJECT_ROOT = Path.cwd().parent
DATA_DIR = Path.cwd().parent / "data"
RAW_FILE = DATA_DIR / "raw" / "variant_summary.txt.gz"
INTERIM_OUT = DATA_DIR / "interim" / "labeled_snvs.parquet"

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

## 02 — Load + label in one call, and SEE the data funnel

`load_and_label_clinvar` applies the three filters (GRCh38 → SNV →
confident binary label) and returns the per-stage counts. We print the funnel
so the data loss is visible, not silent.

In [ ]:
df_labeled, funnel = load_and_label_clinvar(RAW_FILE, return_funnel=True)
print_funnel(funnel)

## 03 — What columns survived, and how complete is each?

A quick completeness scan tells us which columns are dependable enough to
build on. (The columns we rely on downstream — `Name`, `GeneSymbol`,
`ClinicalSignificance` — should be near-fully populated.)

In [ ]:
completeness = df_labeled.notna().mean().sort_values()
completeness.to_frame("non_null_fraction")

## 04 — Class balance

We need the benign/pathogenic split *now*: it decides which metrics are
trustworthy downstream and whether the models need class weighting.

In [ ]:
counts = df_labeled["label"].value_counts().rename({0: "benign", 1: "pathogenic"})
print(counts)
print(f"\nPathogenic fraction: {df_labeled['label'].mean():.1%}")

In [ ]:
fig, ax = plt.subplots(figsize=(4, 3))
counts.plot.bar(ax=ax, color=["#4C72B0", "#C44E52"])
ax.set_title("Confidently labeled GRCh38 SNVs")
ax.set_ylabel("count")
ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()

## 05 — Do these rows carry a protein-level consequence?

Our feature extractor needs the `p.(...)` part of the HGVS `Name`, e.g.
`NM_000546.6(TP53):c.524G>A (p.Arg175His)`. Here we just *confirm the format*
and gauge how many rows even have a protein consequence — the actual
missense isolation happens in notebook 02.

In [ ]:
has_protein = df_labeled["Name"].str.contains(r"p\.", regex=True, na=False)
print(f"Labeled SNVs with a protein-level (p.) consequence: {has_protein.sum():,} "
      f"({has_protein.mean():.1%})")

In [ ]:
df_labeled.loc[has_protein,
               ["GeneSymbol", "Name", "ClinicalSignificance", "label"]].head()

## 06 — Which genes dominate?

A glance at gene frequency foreshadows the **gene-leakage** concern we test in
notebook 03: if a handful of genes supply most variants, a naive random split
can let the model memorize genes instead of learning substitution chemistry.

In [ ]:
df_labeled.loc[has_protein, "GeneSymbol"].value_counts().head(15)

## 07 — Write the interim checkpoint

This is the notebook's hand-off. Everything above was *discovering*; this cell
*produces the artifact* notebook 02 consumes. Parquet preserves dtypes (no
more `dtype=str` re-coercion) and reloads in a fraction of CSV's time.

In [ ]:
INTERIM_OUT.parent.mkdir(parents=True, exist_ok=True)
df_labeled.to_parquet(INTERIM_OUT, index=False)
print(f"Wrote {len(df_labeled):,} labeled SNVs → {INTERIM_OUT.relative_to(PROJECT_ROOT)}")